# Capstone — mirrors your deployed research paper

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/imatiq/ML_Internship/blob/main/work/notebooks/capstone.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/imatiq/ML_Internship"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
else:
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

print("Working dir:", os.getcwd())
assert os.path.exists("data/raw/content_refresh_anonymized.csv"), "starter CSV not found -- are you at the repo root?"
print("Starter data found. You're ready.")

Working dir: /content/flyrank-ml-internship-starter
Starter data found. You're ready.


## 1. Question

*The research question and the decision it supports.*

**Abstract (written last, placed first):** Can FlyRank content pages be grouped into a
small number of interpretable archetypes from observable search/engagement metrics alone,
well enough to prioritize which pages a content strategist reviews first? Using a 30,000-row
anonymized slice of FlyRank's content-performance data, we cluster pages with K-Means (k=6,
chosen by silhouette score) into six archetypes and rank them by decline rate. The resulting
archetype queue does not beat a simple, transparent rule-based baseline (precision@20: 0.50
vs 0.65; precision@50: 0.46 vs 0.50), and degrades further under an honest client-held-out
split (precision@20: 0.35). This is decision-support evidence, not a production-ready score:
the archetypes are moderately stable (ARI=0.59 across reseeds) and useful for grouping pages
into standing review buckets, but a reviewer with limited time should lean on the rule
baseline's reason codes over the archetype ranking alone.

**Decision this supports:** instead of a strategist reviewing 30,000 pages one at a time, they
triage by archetype -- assign each of the six groups a standard playbook (refresh, monitor,
deprioritize...) and route pages by cluster membership, spending limited review time on the
highest-decline-rate groups first.

**Cost of a wrong call:** a misclassified declining page grouped into a low-priority archetype
keeps losing visibility unreviewed; a healthy page wrongly grouped into a high-priority
archetype wastes review time. Because clustering has no ground-truth label, the safeguard is
inspecting real pages per archetype (done in Section 4/ML-08) and disclosing the honest,
degraded, held-out numbers (Section 4/ML-09) rather than the flattering in-sample ones.

In [2]:
print("Research question: do observable-metric archetypes (K-Means) beat a hand rule")
print("for prioritizing which content pages a strategist reviews first?")
print("Decision supported: per-archetype triage playbooks for a 30,000-page review backlog.")

Research question: do observable-metric archetypes (K-Means) beat a hand rule
for prioritizing which content pages a strategist reviews first?
Decision supported: per-archetype triage playbooks for a 30,000-page review backlog.


## 2. Data

*Which release, which tables, date windows, what you excluded and why. Public-safe.*

**Source:** `data/raw/content_refresh_anonymized.csv` -- a single-snapshot, anonymized
slice of FlyRank content-performance data. 30,000 rows, 44 columns, 32 pseudonymous clients.
Unit of analysis: one row = one content page, described by a 90-day rolling window as of
export. This is a snapshot, not a time series -- no repeated daily observations per page.

**Date window:** `content_age_days` ranges 90-564 days; all traffic/engagement fields are
90-day trailing aggregates as of the export date. No absolute calendar dates are used or
needed.

**Features used for clustering** (11 raw candidates from ML-04, 7 kept into the model in
ML-08 after ML-09's leakage/overlap check): `log1p(impressions_90d)`, `ctr`, `avg_position`,
`engagement_rate`, `word_count`, `content_age_days`, `days_since_last_update`.

**Excluded, with why:**
- `trend_direction`, `trend_pct` -- these define the evaluation label (`is_declining_label`);
  using them as features would be reading the answer key.
- `search_volume` -- near-zero correlation (0.001) with actual impressions (ML-02), adds noise
  not signal.
- Pre-bucketed tier columns (`position_tier`, `impression_tier`, ...) -- double-count the raw
  numeric fields already included.
- `provider_used`, `model_used` (which LLM generated an article) -- product/process flags, not
  observable page performance; never used anywhere in scoring.

**Public-safe:** no client names, domains, URLs, page titles, or raw queries anywhere in this
repo (`DATA_USE.md`); IDs are pseudonymous and used for grouping/joins only, never as
clustering features.

In [3]:
import pandas as pd, numpy as np
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
print(f"Rows: {len(df):,}  Columns: {df.shape[1]}  Unique clients: {df['client_id'].nunique()}")
print(f"content_age_days range: {df['content_age_days'].min()}-{df['content_age_days'].max()} days")
features_used = ["impressions_90d", "ctr", "avg_position", "engagement_rate",
                  "word_count", "content_age_days", "days_since_last_update"]
print("Features used:", features_used)

Rows: 30,000  Columns: 44  Unique clients: 32
content_age_days range: 90-564 days
Features used: ['impressions_90d', 'ctr', 'avg_position', 'engagement_rate', 'word_count', 'content_age_days', 'days_since_last_update']


## 3. Methodology

*Assumptions, features, label definition, baseline, validation design, leakage checks.*

**Task type:** clustering (unsupervised) -- there is no pre-existing archetype label; the
goal is groups of pages that behave alike across several metrics at once (ML-03).

**Label/proxy (evaluation only, never a clustering input):** `is_declining_label =
(trend_direction == "down")`. Base rate across the 30,000 rows: 0.542.

**Baseline (ML-07):** a transparent, hand-written rule -- visibility-gated score built from
`stale_but_visible`, `thin_but_visible`, `rankable_position`, and `ranking_low_ctr` reason
codes, no fitted weights. Precision@20 = 0.65, precision@50 = 0.50 (base rate 0.542; the
rule's precision@50 lift over base rate is only +0.041 -- weaker than it looks at a glance).

**Model:** K-Means, `k` swept 2-8 by silhouette score; k=6 chosen at a local peak
(silhouette=0.304) as the smallest k giving distinct, nameable groups. Features standardized
(`StandardScaler`) so `log1p(impressions)` doesn't structurally dominate distance.

**Validation design:** clustering has no future value to hold out, so instead of a
train/test split we ran (a) a **reseed stability check** (refit with a different random seed,
Adjusted Rand Index = 0.59 -- moderate, not strong agreement) and (b) a **client-grouped
holdout**: fit the scaler + K-Means on 22 of 32 pseudonymous clients only, then assign the
remaining 10 clients' pages to the nearest fitted cluster and score with train-derived decline
rates. Honest precision@20 dropped to 0.35 (base rate 0.502) -- clearly worse than the
in-sample 0.50, confirming the reseed instability matters in practice, not just in theory.

**Leakage check:** four of the seven features (`log_impressions`, `ctr`, `avg_position`,
`engagement_rate`) are built from the same 90-day window that overlaps the label's own
last-30-day window. A confession test (refit without those four) showed the decline-rate
spread across clusters drops from 0.505 to 0.364 -- a real but partial dependency, disclosed
rather than hidden.

In [4]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.metrics.cluster import adjusted_rand_score

df["is_declining_label"] = (df["trend_direction"].str.lower() == "down").astype(int)

feat = pd.DataFrame(index=df.index)
feat["log_impressions"] = np.log1p(df["impressions_90d"])
feat["ctr"] = df["ctr"].fillna(0)
feat["avg_position"] = df["avg_position"].replace(0, np.nan)
feat["avg_position"] = feat["avg_position"].fillna(feat["avg_position"].median())
feat["engagement_rate"] = df["engagement_rate"].fillna(0)
feat["word_count"] = df["word_count"].fillna(df["word_count"].median())
feat["content_age_days"] = df["content_age_days"]
feat["days_since_last_update"] = df["days_since_last_update"]

X = StandardScaler().fit_transform(feat)
k = 6
km = KMeans(n_clusters=k, random_state=42, n_init=10).fit(X)
sil = silhouette_score(X, km.labels_, sample_size=8000, random_state=42)
df["cluster"] = km.labels_

km_reseed = KMeans(n_clusters=k, random_state=7, n_init=10).fit(X)
ari = adjusted_rand_score(km.labels_, km_reseed.labels_)

print(f"Baseline (ML-07 rule): base_rate=0.542  P@20=0.65  P@50=0.50")
print(f"k={k} silhouette={sil:.3f}")
print(f"Reseed stability (ARI): {ari:.3f}")

Baseline (ML-07 rule): base_rate=0.542  P@20=0.65  P@50=0.50
k=6 silhouette=0.304
Reseed stability (ARI): 0.590


## 4. Results (vs baseline)

*Model vs baseline on the same split. The honest table.*

| Evaluation | Base rate | Precision@20 | Precision@50 |
|---|---|---|---|
| ML-07 rule baseline | 0.542 | **0.65** | **0.50** |
| K-Means archetype queue -- in-sample | 0.542 | 0.50 | 0.46 |
| K-Means archetype queue -- honest, client-grouped holdout | 0.502 | 0.35 | 0.32 |

The archetype queue loses to the plain rule baseline on both cuts even under the flattering
in-sample evaluation, and degrades further once evaluated the honest way (fit on 22 clients,
score 10 held-out clients never seen during fitting): precision@20 falls from 0.50 to 0.35,
and precision@50 (0.32) sits below the held-out base rate (0.502) -- worse than picking pages
at random on that cut. This lines up with the moderate reseed instability (ARI=0.59): archetype
boundaries don't reform identically on a different slice of clients, so a priority order
learned from training clusters doesn't transfer cleanly.

In [5]:
client_holdout_results = {
    "rule_baseline": {"base_rate": 0.542, "p20": 0.65, "p50": 0.50},
    "kmeans_in_sample": {"base_rate": 0.542, "p20": 0.50, "p50": 0.46},
    "kmeans_honest_holdout": {"base_rate": 0.502, "p20": 0.35, "p50": 0.32},
}
for name, r in client_holdout_results.items():
    print(f"{name:22s} base_rate={r['base_rate']:.3f}  P@20={r['p20']:.2f}  P@50={r['p50']:.2f}")

rule_baseline          base_rate=0.542  P@20=0.65  P@50=0.50
kmeans_in_sample       base_rate=0.542  P@20=0.50  P@50=0.46
kmeans_honest_holdout  base_rate=0.502  P@20=0.35  P@50=0.32


## 5. Limitations

*What this work cannot claim.*

- **Not causal.** This is cross-sectional, single-snapshot data. Nothing here shows that
  refreshing a page *causes* it to stop declining -- only that certain observable metric
  combinations are *associated with* a higher decline rate (`writing-honest-claims` ladder).
- **Not production-ready as scored.** The honest, client-held-out precision (0.35 at K=20)
  is well below the in-sample number (0.50) and below the simple rule baseline (0.65) at every
  cut tested. This queue is decision-support at best, not a number to publish without the
  caveat.
- **Moderate cluster stability, not strong.** ARI=0.59 between reseeds means roughly a third
  of pages could land in a different archetype on a re-run; archetype identity near a boundary
  should not be treated as fixed.
- **Some separation leans on window-overlapping features.** Part (not all) of the modest
  cluster separation depends on features that share the label's own 90-day window
  (`log_impressions`, `ctr`, `avg_position`, `engagement_rate`); the confession test showed
  the spread shrinks (0.505 -> 0.364) but doesn't collapse without them.
- **No trajectory signal.** Static snapshot features cannot see *change* -- a page that just
  started declining looks identical to a page that has been quiet for a year, until the next
  scored batch.
- **Small dataset for some tiers.** `Near-Zero-Traffic` (n=140) and several freshness/reason-
  code buckets in earlier notebooks have n below ~50 in places -- reported but not treated as
  confirmed patterns (ML-06 explicitly flagged the `stale_visible_page` bucket at n=17 as
  insufficient data).

In [6]:
limitations = [
    "not causal -- cross-sectional data only",
    "honest client-holdout precision (0.35 @20) below the rule baseline (0.65) and below in-sample (0.50)",
    "cluster reseed ARI = 0.59 -- moderate stability only",
    "some cluster separation depends on label-window-overlapping features",
    "no trajectory / change signal in a single 90-day snapshot",
    "Near-Zero-Traffic archetype is thin (n=140)",
]
for l in limitations:
    print("-", l)

- not causal -- cross-sectional data only
- honest client-holdout precision (0.35 @20) below the rule baseline (0.65) and below in-sample (0.50)
- cluster reseed ARI = 0.59 -- moderate stability only
- some cluster separation depends on label-window-overlapping features
- no trajectory / change signal in a single 90-day snapshot
- Near-Zero-Traffic archetype is thin (n=140)


## 6. Ranked recommendations

*The action playbook output — the paper's recommendations section.*

From ML-10 (`work/notebooks/w07_action_playbook.ipynb`), the six archetypes map to one
standing action each, ordered by decline rate:

1. **Stale Heavyweights** (n=2,887, decline rate 0.65) -> `refresh` -- long, old-updated,
   still-visible pages; refresh content and update dates first.
2. **Aging Page-One (Engagement Gap)** (n=5,925, 0.62) -> `refresh_title_and_meta` -- ranking
   near page 1 but stale with thin engagement; cheapest high-leverage fix.
3. **Young & Slipping** (n=11,058, 0.61) -> `monitor_closely` -- newer pages already at risk;
   watch, don't rewrite yet.
4. **Established Performers** (n=7,352, 0.41) -> `monitor` -- below base rate; routine
   check-ins only.
5. **Low-Demand Long-Tail** (n=2,638, 0.37) -> `low_priority` -- deep position, low
   impressions; skip unless demand shifts.
6. **Near-Zero-Traffic** (n=140, 0.14) -> `deprioritize` -- almost no impressions; quiet, not
   failing.

**How a FlyRank editor uses this tomorrow:** pull the top of `work/outputs/
archetype_action_playbook.csv`, start with `Stale Heavyweights` and
`Aging Page-One`, and cross-check each pick against the simpler ML-07 rule queue before
committing review time -- the rule caught more true declines at the very top (P@20=0.65) even
though it can't explain *why* a page is grouped the way it is the way the archetype profile
can. Confidence: moderate for archetype-level triage buckets, low for trusting any single
row's rank in isolation (see Section 5).

In [7]:
summary_actions = {
    "Stale Heavyweights": "refresh",
    "Aging Page-One (Engagement Gap)": "refresh_title_and_meta",
    "Young & Slipping": "monitor_closely",
    "Established Performers": "monitor",
    "Low-Demand Long-Tail": "low_priority",
    "Near-Zero-Traffic": "deprioritize",
}
for name, action in summary_actions.items():
    print(f"{name:35s} -> {action}")

Stale Heavyweights                  -> refresh
Aging Page-One (Engagement Gap)     -> refresh_title_and_meta
Young & Slipping                    -> monitor_closely
Established Performers              -> monitor
Low-Demand Long-Tail                -> low_priority
Near-Zero-Traffic                   -> deprioritize


## 7. Artifacts the paper embeds

*Generate/collect the charts and tables your deployed page will show.*

In [9]:
import json
from pathlib import Path

with open("work/playbook_summary.json") as f:
    summary = json.load(f)

print("Chart files ready for the paper:")
for p in sorted(Path("work/outputs/charts").glob("*.svg")):
    print(" -", p)

print()
print("Table: archetype profile")
for row in summary["archetype_profile"]:
    print(f"  {row['archetype']:35s} n={row['n']:6,d}  decline_rate={row['decline_rate']:.3f}")

print()
print("Table: results vs baseline")
print("  rule_baseline          base_rate=0.542  P@20=0.65  P@50=0.50")
print(f"  kmeans_in_sample        base_rate=0.542  P@20=0.50  P@50=0.46")
print(f"  kmeans_honest_holdout   base_rate=0.502  P@20=0.35  P@50=0.32")

Chart files ready for the paper:

Table: archetype profile
  Stale Heavyweights                  n= 2,887  decline_rate=0.647
  Aging Page-One (Engagement Gap)     n= 5,925  decline_rate=0.617
  Young & Slipping                    n=11,058  decline_rate=0.608
  Established Performers              n= 7,352  decline_rate=0.412
  Low-Demand Long-Tail                n= 2,638  decline_rate=0.369
  Near-Zero-Traffic                   n=   140  decline_rate=0.143

Table: results vs baseline
  rule_baseline          base_rate=0.542  P@20=0.65  P@50=0.50
  kmeans_in_sample        base_rate=0.542  P@20=0.50  P@50=0.46
  kmeans_honest_holdout   base_rate=0.502  P@20=0.35  P@50=0.32


## Self-check

Before you submit, confirm each line honestly:

- [X] Every section above is filled — markdown thinking AND the code that backs it
- [X] The notebook runs top to bottom with no errors (Runtime → Run all)
- [X] No client names, URLs, or private queries anywhere
- [X] My claims use careful words: observed, measured, directional, decision-support
- [X] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
- [X] My deployed paper has **all 9 sections** — including the **Abstract** at the top and **Acknowledgments & data credit** (the https://flyrank.ai link) at the bottom.
- [X] **ML-12 done in this notebook's closing cells:** 5-minute demo outline + a social-post cut + a 3-sentence employer-facing summary.


## ML-12 — Closing: demo outline, social cut, employer summary

**5-minute demo outline:**
1. (30s) The problem: 30,000 pages, one strategist, no way to review them all one at a time.
2. (60s) Show the six archetypes and their decline rates (`decline_rate_by_archetype.svg`).
3. (90s) Show the honest results table -- in-sample vs client-held-out -- and explain why the
   held-out number is the one that matters.
4. (60s) Walk one archetype's top rows in `archetype_action_playbook.csv` and its reason.
5. (60s) Limitations in one breath: not causal, moderate stability, doesn't beat the simple
   rule baseline -- so treat it as a second opinion alongside the rule queue, not a replacement.

**Social-post cut (one finding + one chart + one method sentence + link):**
> We clustered 30,000 FlyRank content pages into six behavior archetypes with K-Means --
> the riskiest group ("Stale Heavyweights") declines at 65% vs a 54% dataset base rate, but
> the honest, client-held-out accuracy is lower than a simple hand-written rule. Full method
> and numbers: [deployed paper link].
> ![Decline rate by archetype](/content/flyrank-ml-internship-starter/work/decline_rate_by_archetype.svg)

**Employer-facing 3-sentence summary:**
I built an unsupervised content-triage system for a 30,000-row real (anonymized) SEO content
dataset, clustering pages into six interpretable archetypes with K-Means and validating them
with reseed-stability and client-held-out checks rather than trusting in-sample numbers. The
honest evaluation showed the archetype queue underperforms a simpler rule-based baseline,
which I reported plainly instead of hiding -- and used to write specific, evidence-scoped
recommendations for which pages a content team should review first.